# Verify: AttributeRegistry Edge Cases

**What changed:** 5 fail-fast validations added to AttributeRegistry so errors are caught at registration time, not later in the pipeline.

**Files:** `gbp/core/attributes/spec.py`, `builder.py`, `registry.py`

Each cell tests one behavior. Run top-to-bottom and observe the results.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from gbp.core.attributes.registry import AttributeRegistry
from gbp.core.enums import AttributeKind

print("OK")

## 1. Invalid aggregation

Before: `aggregation="median"` was silently accepted, failed later during spine assembly.  
Now: immediate error at `AttributeSpec` construction.

In [ ]:
reg = AttributeRegistry()

try:
    reg.register(
        name="bad_agg",
        data=pd.DataFrame({"facility_id": ["s1"], "value": [10.0]}),
        entity_type="facility",
        kind=AttributeKind.COST,
        grain=("facility_id",),
        value_column="value",
        aggregation="median",  # <-- invalid
    )
except ValueError as e:
    print(f"Caught: {e}")

Verify that valid aggregations still work:

In [ ]:
from gbp.core.attributes.spec import VALID_AGGREGATIONS

print("Valid aggregations:", sorted(VALID_AGGREGATIONS))

# Check that "sum" works
reg = AttributeRegistry()
reg.register(
    name="ok_sum",
    data=pd.DataFrame({"facility_id": ["s1"], "value": [10.0]}),
    entity_type="facility",
    kind=AttributeKind.COST,
    grain=("facility_id",),
    value_column="value",
    aggregation="sum",
)
print(f"Registered: {reg.names}")

## 2. Duplicate name

Before: second `register()` silently overwrote the first.  
Now: `ValueError` — each attribute needs a unique name.

In [ ]:
reg = AttributeRegistry()
df = pd.DataFrame({"facility_id": ["s1"], "value": [10.0]})

reg.register(
    name="my_cost",
    data=df,
    entity_type="facility",
    kind=AttributeKind.COST,
    grain=("facility_id",),
    value_column="value",
)
print(f"First registration OK: {reg.names}")

try:
    reg.register(
        name="my_cost",  # <-- same name
        data=df,
        entity_type="facility",
        kind=AttributeKind.COST,
        grain=("facility_id",),
        value_column="value",
    )
except ValueError as e:
    print(f"Caught: {e}")

## 3. Non-numeric values

Before: strings in value column were silently coerced to NaN, producing empty spines.  
Now: error at registration.

In [ ]:
reg = AttributeRegistry()

try:
    reg.register(
        name="bad_vals",
        data=pd.DataFrame({"facility_id": ["s1", "s2"], "value": ["abc", "def"]}),
        entity_type="facility",
        kind=AttributeKind.ADDITIONAL,
        grain=("facility_id",),
        value_column="value",
    )
except ValueError as e:
    print(f"Caught: {e}")

All-`None` values are legitimate for nullable attributes (e.g. `reliability` on edges may be unknown):

In [ ]:
reg = AttributeRegistry()

reg.register(
    name="all_none",
    data=pd.DataFrame({"facility_id": ["s1", "s2"], "value": [None, None]}),
    entity_type="facility",
    kind=AttributeKind.ADDITIONAL,
    grain=("facility_id",),
    value_column="value",
    nullable=True,
)
print(f"OK — all None + nullable=True accepted: {reg.names}")

## 4. Empty DataFrame + non-nullable

Before: empty DataFrame was silently registered, failed at build_spines.  
Now: immediate error if `nullable=False`.

In [ ]:
reg = AttributeRegistry()
empty_df = pd.DataFrame({"facility_id": pd.Series([], dtype="str"), "value": pd.Series([], dtype="float64")})

# nullable=False + empty -> error
try:
    reg.register(
        name="empty_required",
        data=empty_df,
        entity_type="facility",
        kind=AttributeKind.CAPACITY,
        grain=("facility_id",),
        value_column="value",
        nullable=False,
    )
except ValueError as e:
    print(f"Caught: {e}")

# nullable=True + empty -> OK
reg.register(
    name="empty_optional",
    data=empty_df,
    entity_type="facility",
    kind=AttributeKind.CAPACITY,
    grain=("facility_id",),
    value_column="value",
    nullable=True,
)
print(f"OK — empty + nullable=True accepted: {reg.names}")

## 5. Improved "no join keys" error message

Before: `"no join keys shared between spine [...] and right [...]"` — unclear what to fix.  
Now: error includes `resolved_grain`, expected merge keys, and hints about grain mismatch / missing time resolution.

This error is hard to trigger naturally (requires a mismatch between spine and prepared attribute data), so we use a mock to demonstrate the message:

In [ ]:
from gbp.core.attributes.builder import AttributeBuilder
from gbp.core.attributes.spec import AttributeSpec
from unittest.mock import patch

spec = AttributeSpec(
    name="cost",
    kind=AttributeKind.COST,
    entity_type="facility",
    grain=("facility_id", "date"),
    resolved_grain=("facility_id", "period_id"),
    value_column="v",
    source_table="t",
)
b = AttributeBuilder("facility")
b.register(spec)

# Patch prepare to return a frame with no overlapping columns
no_overlap = pd.DataFrame({"alien_col": [1.0], "cost": [1.0]})
with patch("gbp.core.attributes.builder._prepare_attribute_frame", return_value=no_overlap):
    try:
        b.build_spines(
            pd.DataFrame({"facility_id": ["a"]}),
            {"cost": pd.DataFrame({"facility_id": ["a"], "period_id": ["p0"], "v": [1.0]})},
        )
    except ValueError as e:
        print("New error message:")
        print(e)

## 6. Full pipeline — sanity check

Build a toy model with a custom registry attribute and verify `build_model` still works end-to-end.

In [ ]:
from datetime import date
from gbp.build.pipeline import build_model
from gbp.core.model import RawModelData
from gbp.core.enums import FacilityRole, PeriodType

raw = RawModelData(
    facilities=pd.DataFrame({
        "facility_id": ["d1", "s1", "s2"],
        "facility_type": ["depot", "station", "station"],
        "name": ["Depot", "St1", "St2"],
    }),
    commodity_categories=pd.DataFrame({
        "commodity_category_id": ["working_bike"], "name": ["Bike"], "unit": ["bike"],
    }),
    resource_categories=pd.DataFrame({
        "resource_category_id": ["truck"], "name": ["Truck"],
        "base_capacity": [20.0], "capacity_unit": ["bike"],
    }),
    planning_horizon=pd.DataFrame({
        "planning_horizon_id": ["h1"], "name": ["H1"],
        "start_date": [date(2025, 1, 1)], "end_date": [date(2025, 1, 4)],
    }),
    planning_horizon_segments=pd.DataFrame({
        "planning_horizon_id": ["h1"], "segment_index": [0],
        "start_date": [date(2025, 1, 1)], "end_date": [date(2025, 1, 4)],
        "period_type": [PeriodType.DAY.value],
    }),
    periods=pd.DataFrame({
        "period_id": ["p0", "p1", "p2"],
        "planning_horizon_id": ["h1"] * 3, "segment_index": [0] * 3,
        "period_index": [0, 1, 2], "period_type": [PeriodType.DAY.value] * 3,
        "start_date": [date(2025, 1, 1), date(2025, 1, 2), date(2025, 1, 3)],
        "end_date": [date(2025, 1, 2), date(2025, 1, 3), date(2025, 1, 4)],
    }),
    facility_roles=pd.DataFrame({
        "facility_id": ["d1", "d1", "s1", "s1", "s1", "s2", "s2"],
        "role": ["storage", "transshipment", "sink", "storage", "source", "sink", "storage"],
    }),
    facility_operations=pd.DataFrame({
        "facility_id": ["d1"] * 3 + ["s1"] * 3 + ["s2"] * 3,
        "operation_type": ["receiving", "storage", "dispatch"] * 3,
        "enabled": [True] * 9,
    }),
    edge_rules=pd.DataFrame({
        "source_type": ["depot"], "target_type": ["station"],
        "commodity_category": ["working_bike"], "modal_type": ["road"], "enabled": [True],
    }),
    edges=pd.DataFrame({
        "source_id": ["d1", "d1"], "target_id": ["s1", "s2"],
        "modal_type": ["road", "road"], "distance": [1.0, 2.0],
        "distance_unit": ["km", "km"], "lead_time_hours": [24.0, 48.0],
        "reliability": [0.99, 0.99],
    }),
    edge_commodities=pd.DataFrame({
        "source_id": ["d1", "d1"], "target_id": ["s1", "s2"],
        "modal_type": ["road", "road"], "commodity_category": ["working_bike"] * 2,
        "enabled": [True, True], "capacity_consumption": [1.0, 1.0],
    }),
    demand=pd.DataFrame({
        "facility_id": ["s1", "s1", "s2"],
        "commodity_category": ["working_bike"] * 3,
        "date": [date(2025, 1, 1), date(2025, 1, 2), date(2025, 1, 1)],
        "quantity": [1.0, 2.0, 3.0], "quantity_unit": ["bike"] * 3,
    }),
    resource_fleet=pd.DataFrame({
        "facility_id": ["d1"], "resource_category": ["truck"], "count": [3],
    }),
    resource_commodity_compatibility=pd.DataFrame({
        "resource_category": ["truck"], "commodity_category": ["working_bike"], "enabled": [True],
    }),
    resource_modal_compatibility=pd.DataFrame({
        "resource_category": ["truck"], "modal_type": ["road"], "enabled": [True],
    }),
)

# Register a custom attribute
raw.attributes.register(
    name="custom_cost",
    data=pd.DataFrame({
        "facility_id": ["d1"],
        "date": ["2025-01-01"],
        "cost_per_unit": [10.0],
    }),
    entity_type="facility",
    kind=AttributeKind.COST,
    grain=("facility_id", "date"),
    value_column="cost_per_unit",
)

resolved = build_model(raw)

print(f"Resolved model built OK")
print(f"Attributes: {resolved.attributes.names}")
print(f"custom_cost data:")
resolved.attributes.get("custom_cost").data

---

**Summary:** 5 edge cases now fail fast at registration instead of silently passing through and breaking later in the pipeline. All existing pipelines work as before.